# MoE_Structures_v2 — overview and setup

Structure-only comparison of routed Transformer structures on synthetic tasks: **Dense, Dense-wide,
MoE, MoH, MoA, MoMHA, MoM, MiTA**, each under three balancing modes (none / load-balance loss /
random warm-up) and with per-structure variants.

| Notebook | What it does |
|---|---|
| 00 | this page: environment, tests |
| 01_data | builds the datasets (E:\data\MoE_Structures_v2) and shows examples |
| 02_models | builds every structure and variant: parameters, one forward pass, schematics |
| 03_screening | runs the dataset screening and picks the dataset(s) |
| 04_main_sweep | structures x balancing modes x 3 seeds |
| 05_variants | per-structure variants x balancing modes |
| 06_analysis | all result tables and figures for the report |
| models/<s>/<s>_analysis | everything about one structure |

One trial from the command line: `python -m experiments.run --structure moe --dataset mqar --balance lb --seed 0`
(see docs/code_map.md for the call chain).

In [1]:
import sys, os, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "backbone").exists():        # works from notebooks/ or models/<s>/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from backbone.paths import DATA_ROOT, RUNS_ROOT, FIG_ROOT
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print("project:", ROOT); print("data:", DATA_ROOT); print("runs:", RUNS_ROOT)

project: /home/claude/work/MoE_Structures_v2
data: /home/claude/work/MoE_Structures_v2/_data
runs: /home/claude/work/MoE_Structures_v2/runs


In [2]:
import torch, platform
print("python", platform.python_version(), "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("threads", torch.get_num_threads())

python 3.13.16 | torch 2.5.0a0+gita8d6afb | cuda False
threads 2


## Tests
Parity with the reference code (needs the reference clones, see references/README.md) and the MiTA equivalence test.

In [3]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(r.stdout[-1500:])

..........                                                               [100%]
=============================== warnings summary ===============================
tests/test_parity.py::test_moe_ffn
  /home/claude/ref/SUT/sut_layer/parallel_linear/parallel_experts/parallel_experts.py:13: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
    @custom_fwd(cast_inputs=torch.float16)

tests/test_parity.py::test_moe_ffn
  /home/claude/ref/SUT/sut_layer/parallel_linear/parallel_experts/parallel_experts.py:43: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
    @custom_bwd

tests/test_parity.py::test_moa_attention
  /home/claude/ref2/MoA/moa_layer/parallel_linear/parallel_experts.py:11: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` inste